# Building AI Agent

## Goal

This notebook is a hands-on journey to build an AI agent from scratch.

Each version introduces one new concept, allowing the agent to evolve step by step while practicing AI agent development.

---

## Version 1

In this version, we build the first version of the agent.

The agent can receive a user request, choose the right tool, extract simple arguments, execute the tool, and store the interaction in memory.

This version keeps the architecture simple.
The agent still executes tools directly.

## 1. Imports

In [1]:
import re

## 2. Tools

In [2]:
# Tools

def greeting(name):
    """Greet the given name."""
    return f"Hello {name.title()}, nice to meet you!"


def addition(a, b):
    """Add two numbers."""
    return a + b


def subtraction(a, b):
    """Subtract two numbers."""
    return a - b


def multiplication(a, b):
    """Multiply two numbers."""
    return a * b


def division(a, b):
    """Divide two numbers."""
    if b == 0:
        return "Error: division by zero!"
    return a / b


def power(a, b):
    """Raise a number to a power."""
    return a ** b

In [3]:
# Tool metadata

tools = {
    "greeting": {
        "function": greeting,
        "parameters": ["name"]
    },
    "addition": {
        "function": addition,
        "parameters": ["a", "b"]
    },
    "subtraction": {
        "function": subtraction,
        "parameters": ["a", "b"]
    },
    "multiplication": {
        "function": multiplication,
        "parameters": ["a", "b"]
    },
    "division": {
        "function": division,
        "parameters": ["a", "b"]
    },
    "power": {
        "function": power,
        "parameters": ["a", "b"]
    }
}

In [4]:
# Tool synonyms

tool_synonyms = {
    "greeting": ["hello", "hi", "hey", "good morning", "good afternoon", "good evening"],
    "addition": ["add", "addition", "sum", "plus", "+"],
    "subtraction": ["subtract", "subtraction", "minus", "take away","-"],
    "multiplication": ["multiply", "multiplication", "times","*"],
    "division": ["divide", "division", "divided by", "/"],
    "power": ["power", "raise", "raised to", "**"]
}

## 3. Parser

In [5]:
# Extract name

def extract_name(text):
    match = re.search(
        r"(?:my name is|i am|i'm|this is)\s+([A-Z][a-z]+)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1)

    words = re.findall(r"\b[A-Z][a-z]+\b", text)

    return words[1] if len(words) > 1 else None

In [6]:
# Extract numbers

def extract_numbers(text):
    numbers = []

    for token in text.split():
        token = re.sub(r"[^\w\s]", "", token)

        if token.isdigit():
            numbers.append(int(token))

    return numbers

In [7]:
# Parser

def parser(tool_name, request):
    if tool_name is None:
        return {}

    parameters = tools[tool_name]["parameters"]

    if parameters == ["name"]:
        name = extract_name(request)
        return {"name": name}

    if parameters == ["a", "b"]:
        numbers = extract_numbers(request)
        return {
            "a": numbers[0] if len(numbers) > 0 else None,
            "b": numbers[1] if len(numbers) > 1 else None
        }

    return {}

## 4. Router

In [8]:
# Router

def choose_tool(request):
    request = request.lower()
    scores = {}

    for tool_name, synonyms in tool_synonyms.items():
        score = 0

        for synonym in synonyms:
            if synonym in request:
                score += 1

        if score > 0:
            scores[tool_name] = score

    if not scores:
        return None

    priority = {
        "greeting": 1,
        "addition": 10,
        "subtraction": 10,
        "multiplication": 10,
        "division": 10,
        "power": 10
    }

    best_tool = max(
        scores,
        key=lambda tool: (scores[tool], priority.get(tool, 0))
    )

    return best_tool

## 5. Agent

In [9]:
# Validate arguments

def arguments_are_valid(arguments):
    if not arguments:
        return False

    for value in arguments.values():
        if value is None:
            return False

    return True

In [10]:
# Agent

memory = []

def agent(request):
    state = {
        "request": request,
        "action": None,
        "arguments": {},
        "result": None
    }

    action = choose_tool(request)
    state["action"] = action

    tool_info = tools.get(action)

    if not tool_info:
        state["result"] = "Tool not found!"
        memory.append(state)
        return state

    state["arguments"] = parser(action, request)

    if not arguments_are_valid(state["arguments"]):
        state["result"] = "Invalid arguments"
    else:
        function = tool_info["function"]
        state["result"] = function(**state["arguments"])

    memory.append(state)
    return state

## 6. Tests

In [11]:
agent("What is 2 + 3?")

{'request': 'What is 2 + 3?',
 'action': 'addition',
 'arguments': {'a': 2, 'b': 3},
 'result': 5}

In [12]:
agent("Hello, what is 2 + 3?")

{'request': 'Hello, what is 2 + 3?',
 'action': 'addition',
 'arguments': {'a': 2, 'b': 3},
 'result': 5}

In [13]:
agent("Hello, my name is luca.")

{'request': 'Hello, my name is luca.',
 'action': 'greeting',
 'arguments': {'name': 'luca'},
 'result': 'Hello Luca, nice to meet you!'}

In [14]:
memory

[{'request': 'What is 2 + 3?',
  'action': 'addition',
  'arguments': {'a': 2, 'b': 3},
  'result': 5},
 {'request': 'Hello, what is 2 + 3?',
  'action': 'addition',
  'arguments': {'a': 2, 'b': 3},
  'result': 5},
 {'request': 'Hello, my name is luca.',
  'action': 'greeting',
  'arguments': {'name': 'luca'},
  'result': 'Hello Luca, nice to meet you!'}]

## Notes

Some test cases were intentionally designed to verify specific parts of the agent.

- `"Hello, what is 2 + 3?"` checks that the router correctly prioritizes the user's main intent (addition) over a greeting.
- `"Hello, my name is luca."` uses a lowercase name on purpose to verify that the tool formats the output (`Luca`) instead of relying on the user's input.
- The memory object stores the state of every interaction, allowing the agent to keep track of previous requests, actions, arguments, and results.

Future versions will introduce new components and gradually evolve the architecture.